# Review local recordings

1. **Build `local_data/metadata.csv`** from every audio file in `AUDIO_DIR`, subfolders included. Set `CHUNK_SECONDS` to split each recording into chunks of that many seconds, or leave it `None` to review whole recordings. Skip this cell to reuse an existing `metadata.csv`.
2. **Load `metadata.csv`** (`id`, `path`, `start_time`, `end_time`) into `jupyter_bioacoustic`.

- **Settings** (audio column, display, output): [`annotator_config/config/local_recordings.yaml`](../annotator_config/config/local_recordings.yaml)
- **Form** (boxes + species dropdown from `species.csv`): [`annotator_config/forms/local_recordings.yaml`](../annotator_config/forms/local_recordings.yaml)
- **Annotations** are saved to `data/local_recordings_annotations.csv`.

Paths are relative to this `notebooks/` folder. The annotator only works in JupyterLab.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf

AUDIO_DIR = "../local_data/audio"  # searched including subfolders; on Windows use /, e.g. "D:/recordings"
CHUNK_SECONDS = None  # e.g. 10 splits each recording into 10 s chunks (the last may be shorter); None = whole recordings

AUDIO_EXTENSIONS = {".wav", ".flac", ".mp3", ".ogg"}
METADATA_CSV = Path("../local_data/metadata.csv")

rows = []
for path in sorted(Path(AUDIO_DIR).rglob("*")):
    if path.suffix.lower() not in AUDIO_EXTENSIONS or path.name.startswith("."):
        continue  # not audio, or a hidden macOS "._" copy
    try:
        duration = round(sf.info(path).duration, 3)
    except sf.LibsndfileError:
        duration = 0
    if duration == 0:
        print(f"Skipped unreadable or empty file: {path}")
        continue
    # The widget plays start_time to end_time (seconds into the file), so a whole recording is 0 to its duration.
    step = CHUNK_SECONDS or duration
    for start in np.arange(0, duration, step):
        rows.append({"path": path.as_posix(), "start_time": round(start, 3), "end_time": round(min(start + step, duration), 3)})
if not rows:
    raise FileNotFoundError(f"No audio files found in {Path(AUDIO_DIR).resolve()}")

metadata = pd.DataFrame(rows)
metadata.insert(0, "id", range(1, len(metadata) + 1))
METADATA_CSV.parent.mkdir(exist_ok=True)
metadata.to_csv(METADATA_CSV, index=False)
print(f"{metadata.path.nunique()} recordings -> {len(metadata)} rows in {METADATA_CSV}")
metadata.head()

138 recordings -> 138 rows in ../local_data/metadata.csv


,id,path,start_time,end_time
0,1,../local_data/audio/nutria/clip_1.wav,0.0,11.778
1,2,../local_data/audio/nutria/clip_10.wav,0.0,5.371
2,3,../local_data/audio/nutria/clip_11.wav,0.0,3.392
3,4,../local_data/audio/nutria/clip_12.wav,0.0,15.170
4,5,../local_data/audio/nutria/clip_13.wav,0.0,6.784


In [2]:
from jupyter_bioacoustic import BioacousticAnnotator

# Needs id, path, start_time and end_time columns (written by the cell above).
ann = BioacousticAnnotator(data="../local_data/metadata.csv", config="../annotator_config/config/local_recordings.yaml")
ann.open()

<IPython.core.display.Javascript object>